In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from util import project_points, scale_intrinsics, inv2x2, eigh_2x2, load_cameras, build_covariance

In [ ]:
def gaussian_rasterization(pos, color, opacity_raw, sigma, c2w, H, W, fx, fy,
                           cx, cy, near=2e-3, far=100, pix_guard=64, 
                           T=16, min_conis=1e-6, chi_square_clip=9.21,
                           alpha_max=0.999, alpha_cutoff=1/255.):
    uv, x, y, z = project_points(pos, c2w, H, W, fx, fy, cx, cy)
    
    device = pos.device
    dt = pos.dtype
    
    u, v = uv[:, 0], uv[:, 1]
    frustum = (
        (u > -pix_guard) 
        & (u < W + pix_guard) 
        & (v > -pix_guard) 
        & (v < H + pix_guard) 
        & (z > near) 
        & (z < far)
    )
    uv = uv[frustum]
    pos = pos[frustum]
    color = color[frustum]
    opacity = torch.sigmoid(opacity_raw[frustum]).squeeze(-1).clamp(0, 0.999)
    x = x[frustum]
    y = y[frustum]
    z = z[frustum]
    sigma = sigma[frustum]
    
    ## Project the covariance
    Rcw = c2w[:3, :3]
    Rwc = Rcw.T
    
    # Eq. 5
    J = torch.zeros((pos.shape[0], 2, 3), device=device, dtype=dt)    
    J[:, 0, 0] = fx / z
    J[:, 1, 1] = fy / z
    J[:, 0, 2] = -fx * x / (z * z)
    J[:, 1, 2] = -fy * y / (z * z)
    
    sigma_camera = Rwc.unsqueeze(0) @ sigma @ Rwc.T.unsqueeze(0)
    sigma_uv = J @ sigma_camera @ J.transpose(1, 2)
            
    # Enforce symmetry
    sigma_uv = 0.5 * (sigma_uv + sigma_uv.transpose(1, 2))
        
    # Clamp projected gaussian ellipse size
    evals, evecs = eigh_2x2(sigma_uv)
    evals = torch.clamp(evals, min=1e-6, max=1e4)
        
    sigma_uv = evecs @ torch.diag_embed(evals) @ evecs.transpose(1, 2)
        
    # Filter NaN and infinity
    keep = torch.isfinite(sigma_uv.reshape(sigma_uv.shape[0], -1)).all(dim=-1)
    
    uv = uv[keep]
    pos = pos[keep]
    color = color[keep]
    opacity = opacity[keep]
    z = z[keep]
    sigma_uv = sigma_uv[keep]
    evals = evals[keep]
    
    z, order = torch.sort(z, descending=False)
    uv = uv[order]
    color = color[order]
    opacity = opacity[order]
    sigma_uv = sigma_uv[order]
    evals = evals[order]
    
    u = uv[:, 0]
    v = uv[:, 1]
    
    # Tiling    
    major_variance = evals[:, 1].clamp(min=1e-12, max=1e4) # [N]
    radius = 3.0 * torch.sqrt(major_variance)
    
    umin = torch.floor(u - radius)
    umax = torch.ceil(u + radius)
    vmin = torch.floor(v - radius)
    vmax = torch.ceil(v + radius)
    
    on_screen = (umax >= 0) & (umin < W) & (vmax >= 0) & (vmin < H)
    if not on_screen.any():
        raise Exception("there are no gaussians on screen")
    
    u, v = u[on_screen], v[on_screen]
    color = color[on_screen]
    opacity = opacity[on_screen]
    sigma_uv = sigma_uv[on_screen]
    umin, umax = umin[on_screen], umax[on_screen]
    vmin, vmax = vmin[on_screen], vmax[on_screen]
    
    umin = umin.clamp(0, W - 1)
    umax = umax.clamp(0, W - 1)
    vmin = vmin.clamp(0, H - 1)
    vmax = vmax.clamp(0, H - 1)
    
    # Tile index for each AABB
    umin_tile = (umin // T).to(torch.int64) # [N]
    umax_tile = (umax // T).to(torch.int64)
    vmin_tile = (vmin // T).to(torch.int64)
    vmax_tile = (vmax // T).to(torch.int64)
    
    # Number of tiles each gaussian intersects
    n_u = umax_tile - umin_tile + 1  # [N] [3, 4, 8, 1, 7, 5]
    n_v = vmax_tile - vmin_tile + 1  # [N] [1, 2, 6, 1, 3, 4]
    
    # Build only the actual gaussian-tile intersections. This keeps memory
    # proportional to K = sum(n_u * n_v), instead of N * max_u * max_v.
    num_tiles_per_gaussian = n_u * n_v # [N]
    num_gaussians = umin_tile.shape[0]
    num_tile_intersections = int(num_tiles_per_gaussian.sum().item())

    gaussian_ids = torch.repeat_interleave(
        torch.arange(num_gaussians, device=device, dtype=torch.int64),
        num_tiles_per_gaussian,
        output_size=num_tile_intersections,
    ) # [K]

    starts_per_gaussian = torch.cumsum(num_tiles_per_gaussian, dim=0)
    starts_per_gaussian = starts_per_gaussian - num_tiles_per_gaussian
    local_tile_ids = (
        torch.arange(num_tile_intersections, device=device, dtype=torch.int64)
        - starts_per_gaussian[gaussian_ids]
    ) # [K]

    # v changes fastest inside each gaussian's tile rectangle.
    local_tile_u = local_tile_ids // n_v[gaussian_ids]
    local_tile_v = local_tile_ids % n_v[gaussian_ids]
    flat_tile_u = umin_tile[gaussian_ids] + local_tile_u
    flat_tile_v = vmin_tile[gaussian_ids] + local_tile_v

    num_tiles_u = (W + T - 1) // T
    flat_tile_id = flat_tile_v * num_tiles_u + flat_tile_u

    index_z_order = torch.arange(num_gaussians, device=device, dtype=torch.int64)
    M = num_gaussians + 1
    comp = flat_tile_id * M  + index_z_order[gaussian_ids]
    comp_sorted, perm = torch.sort(comp)
    gaussian_ids = gaussian_ids[perm]
    tile_ids_1d = torch.div(comp_sorted, M, rounding_mode='floor')
    
    unique_tile_ids, nb_gaussian_per_tile = torch.unique_consecutive(tile_ids_1d, return_counts=True)
    start = torch.zeros_like(unique_tile_ids)
    start[1:] = torch.cumsum(nb_gaussian_per_tile[:-1], dim=0)
    end = start + nb_gaussian_per_tile
    
    inverse_covariance = inv2x2(sigma_uv)
    inverse_covariance[:, 0, 0] = torch.clamp(inverse_covariance[:, 0, 0], min=min_conis)
    inverse_covariance[:, 1, 1] = torch.clamp(inverse_covariance[:, 1, 1], min=min_conis)
    
    final_image = torch.zeros((H * W, 3), device=device, dtype=dt)
    
    # Iterate over tiles
    for tile_id, s0, s1 in zip(unique_tile_ids.tolist(), start.tolist(), end.tolist()):
        txi = tile_id % num_tiles_u
        tyi = tile_id // num_tiles_u
        
        tile_gaussian_ids = gaussian_ids[s0:s1]
        
        x0, y0 = txi * T, tyi * T
        x1, y1 = min((txi + 1) * T, W), min((tyi + 1) * T, H)
        if x0 >= x1 or y0 >= y1:
            continue
        
        xs = torch.arange(x0, x1, device=device, dtype=dt)
        ys = torch.arange(y0, y1, device=device, dtype=dt)
        pu, pv = torch.meshgrid(xs, ys, indexing='xy')
        px_u = pu.reshape(-1)
        px_v = pv.reshape(-1)
        
        pixel_idx_1d = (px_v * W + px_u).to(torch.int64)
        
        gaussian_i_u = u[tile_gaussian_ids]
        gaussian_i_v = v[tile_gaussian_ids]
        gaussian_i_color = color[tile_gaussian_ids]
        gaussian_i_opacity = opacity[tile_gaussian_ids] # [N]
        gaussian_i_inverse_covariance = inverse_covariance[tile_gaussian_ids]
        
        du = px_u.unsqueeze(0) - gaussian_i_u.unsqueeze(-1) # [N, T * T]
        dv = px_v.unsqueeze(0) - gaussian_i_v.unsqueeze(-1) # [N, T * T]
        
        A11 = gaussian_i_inverse_covariance[:, 0, 0].unsqueeze(-1) # [N, 1]
        A12 = gaussian_i_inverse_covariance[:, 0, 1].unsqueeze(-1)
        A22 = gaussian_i_inverse_covariance[:, 1, 1].unsqueeze(-1)
        q = A11 * du * du + 2 * A12 * du * dv + A22 * dv * dv # [N, T * T]
        
        inside = q <= chi_square_clip
        g = torch.exp(-0.5 * torch.clamp(q, max=chi_square_clip)) # [N, T * T]
        g = torch.where(inside, g, torch.zeros_like(g))
                
        alpha_i = (gaussian_i_opacity.unsqueeze(-1) * g).clamp(max=alpha_max) # [N, T * T]
        alpha_i = torch.where(alpha_i >= alpha_cutoff, alpha_i, torch.zeros_like(alpha_i))
        one_minus_alpha_i = 1 - alpha_i
        T_i = torch.cumprod(one_minus_alpha_i, dim=0)
        T_i = torch.concatenate([
            torch.ones((1, alpha_i.shape[-1]), device=device, dtype=dt),
            T_i[:-1]
        ], dim=0)
                
        w = alpha_i * T_i
        tile_color = (w.unsqueeze(-1) * gaussian_i_color.unsqueeze(1)).sum(dim=0) # [T * T, 3]
        
        final_image[pixel_idx_1d] = tile_color
        
    return final_image.reshape((H, W, 3)).clamp(0, 1)

In [ ]:
scene = "bonsai"

device = torch.device("mps")

pos = torch.from_numpy(torch.load('out_bonsai/pos_param.pt', weights_only=False)).to(device)
opacity_raw = torch.from_numpy(torch.load('out_bonsai/alpha_raw_param.pt', weights_only=False)).to(device)
color = torch.sigmoid(0.282 * torch.from_numpy(torch.load('out_bonsai/f_dc.pt', weights_only=False))).to(device)
scale_raw = torch.from_numpy(torch.load('out_bonsai/scale_raw.pt', weights_only=False)).to(device)
rot_raw = torch.from_numpy(torch.load('out_bonsai/rot_raw.pt', weights_only=False)).to(device)

sigma = build_covariance(scale_raw, rot_raw, pos.shape[0])

cam_parameters = np.load(f'out_colmap/{scene}/cam_meta.npy', allow_pickle=True).item()

H = cam_parameters['height']
W = cam_parameters['width']
fx, fy = cam_parameters['fx'], cam_parameters['fy']
cx, cy = W / 2, H / 2

H_scaled = H // 4
W_scaled = W // 4

fx, fy, cx, cy = scale_intrinsics(H_scaled, W_scaled, H, W, fx, fy, cx, cy)

H = H_scaled
W = W_scaled

c2ws, images_paths = load_cameras(f'out_colmap/{scene}/cameras.npy', f'image_data/{scene}/images_2')

CAM_ID = 100

c2w = c2ws[CAM_ID].to(device)
image_path = images_paths[CAM_ID]

gaussian_rasterization(pos, color, opacity_raw, sigma, c2w, H, W, fx, fy, cx, cy)